# Notebook 1 — Segment images with Cellpose 4.0.6 and save masks

## Installation 

### for Mac
Latest stable release is Cellpose 4.0.6, [available via conda-forge](https://anaconda.org/conda-forge/cellpose)

```bash
conda env create -f ./envs/cellpose.yml
conda activate cellpose
```

### for colab

`%pip install "cellpose==4.0.6" "torch" "torchvision" "torchaudio" "scikit-image>=0.22.0" "tqdm>=4.66.0" "pandas>=2.2.0"`

In [ ]:
# %% [markdown]
# # Cellpose-SAM v4 (Apple Silicon) — Full Scientific Outputs
# Segments yellow channel JPEGs and writes:
# - *_masks.tiff (uint16 labels, lossless)
# - *_labels_color.png (random color per label)
# - *_mask_preview.png (binary)
# - *_overlay.png (labels on image)
# - *_boundaries.png (1px edges)
# - *_seg.npz (masks + flows + styles)
# - *_objects.csv (per-object geometry + intensity)
# Per channel:
# - segmentation_summary.csv
# - run_metadata.json

In [ ]:
# Cell 0 — Env check
import sys, platform, torch
print("python:", sys.version.split()[0])
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("mps available:", hasattr(torch.backends, "mps") and torch.backends.mps.is_available())

In [ ]:
# Cell 1 — Imports and device
from pathlib import Path
from typing import List
import os, json, time, warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch

from skimage import io, exposure, color, measure, morphology
from cellpose import models

# optional: silence noisy deprecation prints from internal resizing
warnings.filterwarnings("ignore", message=".*Resizing is depricated.*")

def has_mps() -> bool:
    return hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

device = torch.device("mps") if has_mps() else torch.device("cpu")
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
print("using device:", device)

In [ ]:
# Cell 2 — Paths and parameters (YOUR project)
project_root = Path("/Users/ashi/github/cm4ai_codefest2025")

# Inputs: data/<channel>/*.jpg  (names like ..._yellow.jpg)
img_root   = project_root / "data"
channels: List[str] = ["yellow"]       # reference channel for masks

# Outputs per channel under:
out_root   = project_root / "analysis" / "cellpose_results"

# File types to read
image_exts = [".jpg", ".jpeg", ".tif", ".tiff", ".png"]

# --- Model and key segmentation params ---
pretrained_model    = "cpsam"  # Cellpose-SAM v4 model
use_auto_diameter   = False
diameter            = 80
cellprob_threshold  = -10     # more negative = more permissive
flow_threshold      = 0.3      # >0 emptied masks for you; keep 0
invert              = False
normalize_imgs      = True     # v4 tile normalization
batch_size          = 2        # small batches keep MPS cooler

# Preprocess options
do_rescale_intensity = True
do_clahe             = False
clahe_clip           = 2.0
clahe_tiles          = (8, 8)

print("img_root:", img_root)
print("out_root:", out_root)
print("channels:", channels)
print("model:", pretrained_model)
print("diameter:", (None if use_auto_diameter else diameter))
print("cellprob:", cellprob_threshold, "flow:", flow_threshold, "invert:", invert, "normalize:", normalize_imgs)

In [ ]:
# Cell 3 — Utilities
def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def discover_images(folder: Path, exts) -> list[Path]:
    files = []
    for ext in exts:
        files.extend(sorted(folder.glob(f"*{ext}")))
    # de-dup by stem
    seen, uniq = set(), []
    for p in files:
        if p.stem not in seen:
            uniq.append(p); seen.add(p.stem)
    return uniq

def chunked(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

def preprocess_rgb(img: np.ndarray) -> np.ndarray:
    """Mild per-channel rescale, optional CLAHE."""
    out = img
    if out.ndim == 2:
        arr = exposure.rescale_intensity(out) if do_rescale_intensity else out
        if do_clahe:
            f = exposure.equalize_adapthist(arr, clip_limit=clahe_clip, kernel_size=clahe_tiles)
            arr = (f * np.iinfo(arr.dtype).max).astype(arr.dtype) if np.issubdtype(arr.dtype, np.integer) else (f*255).astype(np.uint8)
        return arr
    out = out.copy()
    if do_rescale_intensity:
        for c in range(out.shape[-1]):
            out[..., c] = exposure.rescale_intensity(out[..., c])
    if do_clahe:
        maxv = np.iinfo(out.dtype).max if np.issubdtype(out.dtype, np.integer) else 255
        for c in range(out.shape[-1]):
            f = exposure.equalize_adapthist(out[..., c], clip_limit=clahe_clip, kernel_size=clahe_tiles)
            out[..., c] = (f * maxv).astype(out.dtype)
    return out

# Robust packers for flows/styles (fixes your ValueError)
def pack_flows_styles(flows_i, styles_i):
    """
    Prepare a dict for np.savez_compressed with separate keys per flow component.
    Handles dict/tuple/list/array; falls back to object arrays if needed.
    """
    payload = {}
    try:
        if isinstance(flows_i, dict):
            for k, v in flows_i.items():
                payload[f"flow_{k}"] = np.asarray(v)
        elif isinstance(flows_i, (list, tuple)):
            for i, comp in enumerate(flows_i):
                payload[f"flow_{i}"] = np.asarray(comp)
        else:
            payload["flows"] = np.asarray(flows_i)
    except Exception:
        payload["flows_object"] = np.array(flows_i, dtype=object)

    try:
        payload["styles"] = np.asarray(styles_i, dtype=np.float32)
    except Exception:
        payload["styles_object"] = np.array(styles_i, dtype=object)

    return payload

def save_all_artifacts(im, m, flows_i, styles_i, out_dir: Path, stem: str):
    """
    Writes:
      - uint16 label TIFF
      - colored labels PNG
      - binary preview PNG
      - overlay PNG
      - boundaries PNG
      - NPZ bundle with masks + per-component flows + styles
      - per-object CSV (geometry + intensity)
    """
    ensure_dir(out_dir)

    # 1) label TIFF
    io.imsave(out_dir / f"{stem}_masks.tiff", m.astype(np.uint16), check_contrast=False)

    # 2) colored labels
    if m.max() > 0:
        rng = np.random.default_rng(42)
        rand_colors = rng.uniform(0, 1, size=(m.max()+1, 3))
        colored = color.label2rgb(m, colors=rand_colors, bg_label=0)
        io.imsave(out_dir / f"{stem}_labels_color.png", (colored*255).astype(np.uint8), check_contrast=False)
    else:
        base = im.astype(np.float32); 
        if base.max() > 0: base /= base.max()
        io.imsave(out_dir / f"{stem}_labels_color.png", (base*255).astype(np.uint8), check_contrast=False)

    # 3) binary preview + overlay
    bin_prev = (m > 0).astype(np.uint8) * 255
    io.imsave(out_dir / f"{stem}_mask_preview.png", bin_prev, check_contrast=False)

    base = im.astype(np.float32)
    if base.max() > 0: base /= base.max()
    overlay = color.label2rgb(m, image=base, bg_label=0, alpha=0.20)
    io.imsave(out_dir / f"{stem}_overlay.png", (overlay*255).astype(np.uint8), check_contrast=False)

    # 4) boundaries
    if m.max() > 0:
        edges = morphology.binary_dilation(m > 0, morphology.disk(1)) ^ (m > 0)
        io.imsave(out_dir / f"{stem}_boundaries.png", (edges.astype(np.uint8)*255), check_contrast=False)
    else:
        io.imsave(out_dir / f"{stem}_boundaries.png", np.zeros(m.shape, np.uint8), check_contrast=False)

    # 5) NPZ bundle
    payload = pack_flows_styles(flows_i, styles_i)
    np.savez_compressed(out_dir / f"{stem}_seg.npz", masks=m.astype(np.int32), **payload)

    # 6) per-object CSV
    meas_img = im.mean(axis=-1) if im.ndim == 3 else im
    props = measure.regionprops(m, intensity_image=meas_img)
    rows = []
    for pr in props:
        minr, minc, maxr, maxc = pr.bbox
        rows.append({
            "label": pr.label,
            "area_px": int(pr.area),
            "perimeter_px": float(pr.perimeter),
            "centroid_r": float(pr.centroid[0]),
            "centroid_c": float(pr.centroid[1]),
            "bbox_minr": int(minr), "bbox_minc": int(minc),
            "bbox_maxr": int(maxr), "bbox_maxc": int(maxc),
            "mean_intensity": float(pr.mean_intensity),
            "max_intensity": float(getattr(pr, "max_intensity", np.nan)),
            "min_intensity": float(getattr(pr, "min_intensity", np.nan)),
        })
    pd.DataFrame(rows).to_csv(out_dir / f"{stem}_objects.csv", index=False)

In [ ]:
# Cell 4 — Load Cellpose v4 model (Apple Silicon)
model = models.CellposeModel(
    gpu=False,                     # do not force CUDA; MPS used via 'device'
    pretrained_model=pretrained_model,
    device=device
)
print("loaded model:", model.pretrained_model)

In [ ]:
# Cell 5 — Segmentation with full scientific outputs
for ch in channels:
    in_dir  = img_root / ch
    ch_root = out_root / ch
    png_dir = ch_root / "masks"
    ensure_dir(png_dir)

    files = discover_images(in_dir, image_exts)
    if not files:
        print(f"[{ch}] no images in {in_dir}")
        continue

    print(f"[{ch}] {len(files)} images -> {png_dir}")

    summary_rows, total, failed = [], 0, []

    # Run metadata for reproducibility
    run_meta = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "model": pretrained_model,
        "device": str(device),
        "use_auto_diameter": bool(use_auto_diameter),
        "diameter": (None if use_auto_diameter else float(diameter)),
        "cellprob_threshold": float(cellprob_threshold),
        "flow_threshold": float(flow_threshold),
        "invert": bool(invert),
        "normalize": bool(normalize_imgs),
        "batch_size": int(batch_size),
        "image_exts": image_exts,
    }
    ensure_dir(ch_root)
    with open(ch_root / "run_metadata.json", "w") as f:
        json.dump(run_meta, f, indent=2)

    # process in small batches
    for group in tqdm(list(chunked(files, batch_size)), desc=f"Cellpose [{ch}]"):
        raw_imgs = [io.imread(p) for p in group]
        imgs = [preprocess_rgb(im) for im in raw_imgs]

        try:
            masks_list, flows, styles = model.eval(
                imgs,
                diameter=(None if use_auto_diameter else float(diameter)),
                batch_size=len(imgs),
                channel_axis=-1,          # RGB inputs
                invert=invert,
                normalize=normalize_imgs, # v4 tile normalization
                flow_threshold=flow_threshold,
                cellprob_threshold=cellprob_threshold,
            )
        except Exception as e:
            failed.extend([(p.name, str(e)) for p in group])
            # write placeholders for traceability
            for p, im in zip(group, raw_imgs):
                stem = p.stem
                io.imsave(png_dir / f"{stem}_masks.tiff", np.zeros(im.shape[:2], np.uint16), check_contrast=False)
                io.imsave(png_dir / f"{stem}_mask_preview.png", np.zeros(im.shape[:2], np.uint8), check_contrast=False)
                base = preprocess_rgb(im).astype(np.float32)
                if base.max() > 0: base /= base.max()
                io.imsave(png_dir / f"{stem}_overlay.png", (base*255).astype(np.uint8), check_contrast=False)
            continue

        # save artifacts and accumulate summary
        for idx, (p, im) in enumerate(zip(group, raw_imgs)):
            m = masks_list[idx]
            f = flows[idx] if isinstance(flows, (list, tuple)) else flows
            s = styles[idx] if isinstance(styles, (list, tuple)) else styles

            save_all_artifacts(im, m, f, s, png_dir, p.stem)

            # per-image summary
            n_labels = int(m.max())
            meas_img = im.mean(axis=-1) if im.ndim == 3 else im
            props = measure.regionprops(m, intensity_image=meas_img)
            areas = np.array([pr.area for pr in props], dtype=float) if props else np.array([])
            summary_rows.append({
                "file": p.name,
                "n_labels": n_labels,
                "area_mean": float(areas.mean()) if areas.size else 0.0,
                "area_median": float(np.median(areas)) if areas.size else 0.0,
                "area_min": int(areas.min()) if areas.size else 0,
                "area_max": int(areas.max()) if areas.size else 0,
            })
            total += 1

    # write per-channel summary
    pd.DataFrame(summary_rows).to_csv(ch_root / "segmentation_summary.csv", index=False)
    print(f"[{ch}] done. images segmented: {total}, failures: {len(failed)}")
    if failed:
        print("first failures:", failed[:3])